In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import joblib

from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pystac.client import Client
from odc.stac import load
import xarray as xr

from utils import mask_with_gebco, make_indices

In [ ]:
MAX_DEPTH = -40

# Get all the csvs in the data directory
data_dir = Path("data")
regions = gpd.read_file('postcards.geojson')

# Read them and merge them into a single dataframe
all = []
atolls = []
islands = []

for region in regions.itertuples():
    csv = data_dir / f"{region.name}_training_data.csv"
    is_atoll = region.type == "Atoll"
    print(f"Reading {csv} ({'Atoll' if is_atoll else 'Island'})")

    gdf = gpd.read_file(csv)

    # Make sure everything can be converted to a float
    for col in gdf.columns:
        gdf[col] = gdf[col].astype(float)

    # Replace infinite values with NaN
    gdf = gdf.replace([float('-inf'), float('inf')], float('nan'))

    # Drop rows with missing values
    gdf = gdf.dropna()
    gdf = gdf[gdf.depth > MAX_DEPTH]

    # Get them all and put them in lists
    all.append(gdf)
    if is_atoll:
        atolls.append(gdf)
    else:
        islands.append(gdf)

all_data = gpd.GeoDataFrame(pd.concat(all, ignore_index=True))
atolls_data = gpd.GeoDataFrame(pd.concat(atolls, ignore_index=True))
islands_data = gpd.GeoDataFrame(pd.concat(islands, ignore_index=True))

print(f"\nTotal data points: {len(all_data)}, Atolls: {len(atolls_data)}, Islands: {len(islands_data)}")

In [ ]:
# # Run the training using a combination of adaboost, random forest and gradient boosting
# # along with all data, atolls and islands only and store the results in a dictionary
# results = {}
# for data, name in [(all_data, 'All'), (atolls_data, 'Atolls'), (islands_data, 'Islands')]:
#     print(f"\n{name} data")
#     X = data.drop(columns=['x', 'y', 'depth'])
#     y = data['depth']

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#     for model in [AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor]:
#         model_name = model.__name__
#         print(f"- {model_name}")
#         reg = model()
#         reg.fit(X_train, y_train)

#         y_pred = reg.predict(X_test)
#         mse = mean_squared_error(y_test, y_pred)
#         mae = mean_absolute_error(y_test, y_pred)

#         print(f"- MSE: {mse}")
#         print(f"- MAE: {mae}")

#         results[f"{name} {model_name}"] = {
#             'mse': mse,
#             'mae': mae
#         }

## Results

| Model | MSE | MAE |
| --- | --- | --- |
| All AdaBoostRegressor | 57.399 | 6.307 |
| All RandomForestRegressor | 13.796 | 2.308 |
| All GradientBoostingRegressor | 20.365 | 3.009 |
| Atolls AdaBoostRegressor | 29.731 | 4.508 |
| Atolls RandomForestRegressor | 10.195 | 2.244 |
| Atolls GradientBoostingRegressor | 15.044 | 2.825 |
| Islands AdaBoostRegressor | 44.185 | 4.850 |
| Islands RandomForestRegressor | 18.380 | 2.368 |
| Islands GradientBoostingRegressor | 23.019 | 2.843 |

In [ ]:
# # Print the results as a markdown table
# print("\nResults")
# print("| Model | MSE | MAE |")
# print("| --- | --- | --- |")
# for key, value in results.items():
#     print(f"| {key} | {value['mse']:0.3f} | {value['mae']:0.3f} |")

In [ ]:
data = all_data

train, test = train_test_split(data, test_size=0.3)

depth = train["depth"]
variables = train.drop(columns=["depth", "x", "y"])

# Define the model
# regressor = AdaBoostRegressor()
regressor = RandomForestRegressor()
# regressor = GradientBoostingRegressor()

# Train the model
model = regressor.fit(variables, depth)

# Evaluate on our test data
test_depth = test["depth"]
test_variables = test.drop(columns=["depth", "x", "y"])

predictions = model.predict(test_variables)
mse = mean_squared_error(test_depth, predictions)
mae = mean_absolute_error(test_depth, predictions)

print(f"Mean squared error: {mse:.3f}")
print(f"Mean absolute error: {mae:.3f}")

In [ ]:
# # Write out the model
# joblib.dump(model, "models/2025_03_randomforest_all.joblib")


In [ ]:
# Bounding box for Tuvalu
# bbox = [179.020, -8.665, 179.218, -8.413]

# Bounding box for Suva
bbox = [178.400, -18.200, 178.600, -18.000]

# Test the model on a region we haven't trained on
catalog = Client.open("https://stac.digitalearthpacific.org")
collection = "dep_s2_geomad"

items = catalog.search(collections=[collection], bbox=bbox, datetime="2024").item_collection()
print(f"Found {len(items)} items")

geomad = load(items, bbox=bbox, chunks={})
geomad = make_indices(geomad).compute()

# Do the prediction
# Convert to a stacked array of observations
stacked_arrays = geomad.to_array().stack(dims=["y", "x"])

# Replace any infinities with NaN
stacked_arrays = stacked_arrays.where(stacked_arrays != float("inf"))
stacked_arrays = stacked_arrays.where(stacked_arrays != float("-inf"))

# Replace any NaN values with 0
stacked_arrays = stacked_arrays.squeeze().fillna(0).transpose()

# Predict the classes
predicted = model.predict(stacked_arrays)

# Reshape back to the original 2D array
array = predicted.reshape(geomad.y.size, geomad.x.size)

# Convert to an xarray again, because it's easier to work with
depth_predicted = xr.DataArray(
    array, coords={"y": geomad.y, "x": geomad.x}, dims=["y", "x"]
)

depth = depth_predicted.to_dataset(name="elevation")
depth["elevation_masked"] = mask_with_gebco(depth, -1000, interpolate=False).elevation

depth

In [ ]:
geomad

In [ ]:
from odc.algo import mask_cleanup

land = (geomad.mndwi + geomad.ndwi).squeeze() > 10
land_mask = mask_cleanup(land, [["dilation", 5], ["erosion", 10]])

In [ ]:
import folium
from ipyleaflet import basemaps

depth_masked = depth.where(~land_mask)

centroid = depth_masked.odc.geobox.geographic_extent.centroid.to_crs("EPSG:4326").coords[0]
m = folium.Map(location=[centroid[1], centroid[0]], zoom_start=12, tiles=basemaps.Esri.WorldImagery)

options = {
    "min": -40,
    "max": 0,
    "cmap": "Blues_r"
}
depth_masked.elevation.odc.add_to(m, **options, name="depth")
depth_masked.elevation_masked.odc.add_to(m, **options, name="depth_masked")

# Add layer control
folium.LayerControl().add_to(m)

m

In [ ]:
import folium

depth_predicted_masked = mask_with_gebco(depth_predicted, -1000, interpolate=False)

centroid = location=depth_predicted_masked.odc.geobox.geographic_extent.centroid.to_crs("EPSG:4326").coords[0]
m = folium.Map(location=[centroid[1], centroid[0]], zoom_start=12)

options = {
    "min": -40,
    "max": 0,
    "cmap": "Blues"
}
depth.elevation.odc.add_to(m, **options, name="depth")
depth.elevation_masked.odc.add_to(m, **options, name="depth_masked")

# Add layer control
folium.LayerControl().add_to(m)

m

In [ ]:
depth.elevation.odc.write_cog("elevation_atolls.tif")
# depth.elevation_masked.odc.write_cog("elevation_masked.tif")